<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

<h1 align="center"><b>Laboratorio 5 -- Clasificacion con Datos Desbalanceados</b></h1>

</div>

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

Los datos desbalanceados ocurren cuando una clase tiene muchas mas observaciones que otra. En este problema de deteccion de diabetes, el **86.1%** de los registros corresponden a personas sin diabetes y solo el **13.9%** son casos positivos. Este desbalance provoca que los modelos aprendan a favorecer la clase mayoritaria, obteniendo metricas aparentemente altas de **Accuracy** pero con bajo desempeno en la deteccion de diabeticos.

**Estrategia adoptada:** configuraciones directas y robustas basadas en experiencia practica,
priorizando modelos de ensamble eficientes y tecnicas de balanceo probadas en datos medicos.

Documentacion de referencia: [imbalanced-learn](https://imbalanced-learn.org/stable/index.html)

</div>

---

## 0. Librerias

In [ ]:
import warnings
import pandas as pd
import numpy as np
import os
import joblib
from pathlib import Path

# Visualizacion
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Preprocesamiento
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_auc_score, precision_score, recall_score, f1_score,
    precision_recall_curve, roc_curve, auc as sklearn_auc
)

# Modelos
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Tecnicas de balanceo -- imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120})
SEED = 123
print("Librerias cargadas correctamente.")

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

<h2>1. Comprension del Problema</h2>

**Variable objetivo:** `Diabetes_binary`
- `0` = Sin diabetes
- `1` = Prediabetes o diabetes

**Contexto:** Dataset BRFSS 2015 del CDC con 253.680 adultos estadounidenses. El objetivo es detectar tempranamente la diabetes a partir de 21 indicadores de salud, estilo de vida y factores socioeconomicos.

**Metrica prioritaria:** En deteccion de enfermedades, minimizar los **Falsos Negativos** es critico -- un diabetico no detectado puede sufrir complicaciones graves. Por eso priorizamos **Recall > F1-Score > ROC-AUC > Precision** sobre Accuracy.

</div>

---

## 2. Configuracion de Rutas

In [ ]:
mainpath = (
    "/workspaces/ml-project_analitica_datosv/"
    "ml-proyecto_analitica_datos/data/raw/dataset_clasificacion"
)
filename = "diabetes_binary_health_indicators_BRFSS2015.csv"
fullpath = os.path.join(mainpath, filename)

MODELS_DIR = Path(
    "/workspaces/ml-project_analitica_datosv/ml-proyecto_analitica_datos/models"
)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Lectura de Datos

In [ ]:
data = pd.read_csv(fullpath, sep=",")
pd.set_option("display.max_columns", None)
data.sample(5)

In [ ]:
data.info()

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 4. Exploracion Inicial -- Desbalanceo de Clases

</div>

In [ ]:
counts = data["Diabetes_binary"].value_counts()
pcts   = data["Diabetes_binary"].value_counts(normalize=True) * 100

fig = px.bar(
    x=["Sin diabetes (0)", "Diabetes/Prediabetes (1)"],
    y=counts.values,
    text=[f"{v:,}<br>({p:.1f}%)" for v, p in zip(counts.values, pcts.values)],
    color=["Sin diabetes (0)", "Diabetes/Prediabetes (1)"],
    color_discrete_sequence=["#4C9BE8", "#E87D4C"],
    title="Distribucion de la Variable Objetivo -- Diabetes_binary",
    template="simple_white"
)
fig.update_traces(textposition="outside")
fig.update_layout(title_x=0.5, showlegend=False,
                  yaxis_title="Numero de registros", xaxis_title="Clase")
fig.show()

print(f"Clase 0 (sin diabetes):       {counts[0]:>7,}  ({pcts[0]:.1f}%)")
print(f"Clase 1 (diabetes/prediab.):  {counts[1]:>7,}  ({pcts[1]:.1f}%)")
print(f"Razon de desbalanceo: {counts[0]/counts[1]:.1f}:1")

In [ ]:
missing = data.isnull().sum()
if missing.sum() == 0:
    print("No hay valores faltantes.")
else:
    print(missing[missing > 0])

data.describe().T.round(2)

In [ ]:
corr = data.corr()["Diabetes_binary"].drop("Diabetes_binary").sort_values()

fig = px.bar(
    x=corr.values,
    y=corr.index,
    orientation="h",
    color=corr.values,
    color_continuous_scale="RdBu_r",
    title="Correlacion de cada variable con Diabetes_binary",
    template="simple_white"
)
fig.update_layout(title_x=0.5, height=600,
                  xaxis_title="Correlacion de Pearson", yaxis_title="")
fig.show()

---
## 5. Preparacion de los Datos

In [ ]:
target = "Diabetes_binary"

X = data.drop(columns=[target])
y = data[target]

print(f"X shape: {X.shape}")
print(f"y distribucion:\n{y.value_counts()}")

In [ ]:
# Variables con rango amplio que se benefician del escalamiento
numeric_features = ["BMI", "MentHlth", "PhysHlth"]

# Variables binarias y ordinales
ordinal_features = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk",
    "Sex", "GenHlth", "Age", "Education", "Income"
]

all_features = numeric_features + ordinal_features

print(f"Variables numericas ({len(numeric_features)}): {numeric_features}")
print(f"Variables ordinales/binarias ({len(ordinal_features)}): {ordinal_features}")

In [ ]:
# Preprocesador con escalamiento (para modelos lineales)
numeric_transformer_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

passthrough_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor_scaled = ColumnTransformer(transformers=[
    ("num_scaled", numeric_transformer_scaled, numeric_features),
    ("ordinal",    passthrough_transformer,    ordinal_features)
])

# Preprocesador sin escalamiento (para modelos de arbol/boosting)
preprocessor_no_scaled = ColumnTransformer(transformers=[
    ("pass", SimpleImputer(strategy="median"), all_features)
])

print("Preprocesadores definidos.")

In [ ]:
# Division 70/30 estratificada -- garantiza misma proporcion de clases en ambos sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=SEED
)

print(f"Train: {X_train.shape[0]:,} muestras  |  Test: {X_test.shape[0]:,} muestras")
print(f"Proporcion clase 1 -- Train: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}")

---
## 6. Funcion de Evaluacion

Metricas reportadas por orden de prioridad clinica: **Recall > F1 > ROC-AUC > Precision > Accuracy**

In [ ]:
def evaluate_model(model, model_name):
    print(f"=== {model_name} ===")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    print("\nClassification Report\n")
    print(classification_report(y_test, y_pred,
          target_names=["Sin diabetes", "Diabetes"]))

    # Matriz de confusion
    cm = confusion_matrix(y_test, y_pred)
    vp, fp, fn, vn = cm[1,1], cm[0,1], cm[1,0], cm[0,0]

    cm_df = pd.DataFrame(
        [[vp, fp], [fn, vn]],
        index=["Predicho Positivo", "Predicho Negativo"],
        columns=["Real Positivo", "Real Negativo"]
    )
    labels = [
        [f"VP<br>{vp}", f"FP<br>{fp}"],
        [f"FN<br>{fn}", f"VN<br>{vn}"]
    ]

    fig = px.imshow(
        cm_df, text_auto=False,
        color_continuous_scale=[[0.0,"#7f0000"],[0.5,"#1a1a1a"],[1.0,"#006400"]],
        aspect="auto", title=f"Matriz de Confusion - {model_name}"
    )
    for i in range(2):
        for j in range(2):
            fig.add_annotation(
                x=cm_df.columns[j], y=cm_df.index[i],
                text=labels[i][j], showarrow=False,
                font=dict(size=22, color="white")
            )
    fig.update_layout(width=700, height=500, title_x=0.5,
                      template="plotly_dark", coloraxis_showscale=False)
    fig.update_xaxes(title="Clases reales", side="top")
    fig.update_yaxes(title="Clases predichas")
    fig.show()

    metrics = {
        "Model":     model_name,
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1":        round(f1_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Accuracy":  round((y_pred == y_test).mean(), 4),
    }
    if y_prob is not None:
        metrics["ROC_AUC"] = round(roc_auc_score(y_test, y_prob), 4)

    metrics_df = pd.DataFrame([metrics])
    print("\nMETRICAS (orden de prioridad clinica)\n")
    display(metrics_df.style.hide(axis="index"))
    return metrics

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 7. Modelo Base -- Referencia sin Balanceo

Regresion Logistica estandar sin ninguna tecnica de balanceo.
Sus metricas sirven como **linea base** para cuantificar la mejora real
de cada tecnica optimizada. Se espera Recall bajo en la clase minoritaria
(diabeticos), ya que el modelo aprende a predecir siempre la clase mayoritaria.

</div>

In [ ]:
baseline_model = ImbPipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("model", LogisticRegression(max_iter=500, random_state=SEED))
])

metrics_baseline = evaluate_model(baseline_model, "Baseline -- Sin balanceo")

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 8. Modelos Optimizados -- Estrategia Directa

Para una **ratio de desbalanceo de 6.2:1** en datos medicos, se aplican directamente
tres estrategias de alta eficacia basadas en configuraciones robustas de la literatura
de ML clinico. **Sin busquedas exhaustivas de hiperparametros.**

| Estrategia | Modelo | Mecanismo de balanceo | Ventaja clinica |
|---|---|---|---|
| **8.1** | XGBoost + `scale_pos_weight` | Ponderacion interna en la funcion de perdida | Trabaja con datos reales, sin sinteticos |
| **8.2** | SMOTE + XGBoost | Sobremuestreo sintetico previo al modelo | Aprende sobre distribucion balanceada |
| **8.3** | Balanced Random Forest | Submuestreo automatico por arbol | Robusto, interpretable, sin sinteticos |

> **Por que no GridSearch masivo?** Con 253K registros, un grid de 3 parametros x 5 folds
> implica 15+ entrenamientos de modelos pesados (~horas). Los hiperparametros elegidos
> son configuraciones probadas en practica. El **ajuste de umbral** (Seccion 10) ofrece
> mayor ganancia clinica que el tuning fino de hiperparametros.

</div>

### 8.1 XGBoost con `scale_pos_weight` -- Balanceo Nativo en Boosting

**Tecnica:** `scale_pos_weight = n_negativos / n_positivos ≈ 6.2` pondera la clase minoritaria
directamente en la funcion de perdida del algoritmo de boosting.

**Ventaja en datos medicos:** No genera muestras sinteticas -- trabaja exclusivamente con
registros reales, lo cual es preferido en contextos clinicos donde la calidad del dato es critica.

**Hiperparametros seleccionados:**
- `n_estimators=300, learning_rate=0.05`: convergencia lenta y robusta, evita sobreajuste
- `max_depth=6`: profundidad media, captura interacciones sin memorizar ruido
- `subsample=0.8, colsample_bytree=0.8`: regularizacion estocastica (similar a dropout)
- `min_child_weight=5`: evita particiones en nodos con muy pocas muestras minoritarias

In [ ]:
ratio_classes = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Ratio clases (scale_pos_weight): {ratio_classes:.2f}")

xgb_optimized = ImbPipeline(steps=[
    ("preprocessor", preprocessor_no_scaled),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=ratio_classes,
        min_child_weight=5,
        gamma=0.1,
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
        eval_metric="logloss"
    ))
])

metrics_xgb_opt = evaluate_model(xgb_optimized, "XGBoost + scale_pos_weight")

### 8.2 SMOTE + XGBoost -- Sobremuestreo Sintetico + Boosting

**Tecnica:** SMOTE genera muestras sinteticas de la clase minoritaria interpolando
entre `k=5` vecinos mas cercanos en el espacio de features. El dataset resultante
queda balanceado 1:1 *antes* de entrenar XGBoost.

**Por que SMOTE y no ADASYN?** Con las 18 variables binarias/ordinales del dataset,
SMOTE produce interpolaciones mas estables. ADASYN puede generar ruido concentrado
en zonas de alta densidad con features discretas, sesgando innecesariamente la frontera.

**Diferencia con 8.1:** El modelo no usa `scale_pos_weight` porque el dataset
ya esta balanceado por SMOTE antes del entrenamiento.

In [ ]:
smote_xgb = ImbPipeline(steps=[
    ("preprocessor", preprocessor_no_scaled),
    ("sampler", SMOTE(random_state=SEED, k_neighbors=5)),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        gamma=0.1,
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
        eval_metric="logloss"
    ))
])

metrics_smote_xgb = evaluate_model(smote_xgb, "SMOTE + XGBoost")

### 8.3 Balanced Random Forest -- Ensamble con Submuestreo Interno

**Tecnica:** `BalancedRandomForestClassifier` aplica submuestreo automatico
dentro de cada arbol del bosque: usa **todas** las muestras de la clase minoritaria
y una muestra aleatoria equivalente de la mayoritaria.

| Clase | Dataset completo (train) | Dataset por arbol |
|---|---|---|
| Sin diabetes (0) | ~178,000 | ~25,000 |
| Diabetes (1) | ~25,000 | ~25,000 |

**Ventajas para datos medicos:**
- No genera datos sinteticos (conservador y seguro para registros de salud)
- `feature_importances_` altamente interpretables para equipos clinicos
- Robusto al ruido y outliers en variables como BMI o MentHlth

**Hiperparametros:**
- `n_estimators=300`: bosque suficientemente grande para estabilizar importancias
- `max_depth=20`: permite capturar interacciones complejas entre factores de riesgo
- `min_samples_leaf=3`: evita hojas con muy pocas muestras (sobreajuste)

In [ ]:
brf_optimized = ImbPipeline(steps=[
    ("preprocessor", preprocessor_no_scaled),
    ("model", BalancedRandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=3,
        replacement=True,
        sampling_strategy="auto",
        random_state=SEED,
        n_jobs=-1
    ))
])

metrics_brf_opt = evaluate_model(brf_optimized, "Balanced Random Forest")

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 9. Validacion Cruzada -- Comparacion de Modelos

`StratifiedKFold(k=5)` sobre `X_train` estima la capacidad de generalizacion.
El estrato garantiza la misma proporcion de clases diabeticos/sanos en cada pliegue.

**Ranking:** F1-Score (equilibra Recall y Precision).  
**Metrica clinica principal reportada:** Recall (minimiza falsos negativos).

</div>

In [ ]:
cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

models_cv = {
    "Baseline (LR sin balanceo)" : baseline_model,
    "XGBoost + scale_pos_weight" : xgb_optimized,
    "SMOTE + XGBoost"            : smote_xgb,
    "Balanced Random Forest"     : brf_optimized,
}

results = []
for name, model in models_cv.items():
    print(f"  Evaluando: {name} ...", end=" ", flush=True)
    try:
        scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        results.append({
            "Modelo"   : name,
            "Recall"   : round(np.mean(scores["test_recall"]), 4),
            "F1"       : round(np.mean(scores["test_f1"]), 4),
            "ROC_AUC"  : round(np.mean(scores["test_roc_auc"]), 4),
            "Precision": round(np.mean(scores["test_precision"]), 4),
            "Accuracy" : round(np.mean(scores["test_accuracy"]), 4),
        })
        r = results[-1]
        print(f"Recall={r['Recall']:.4f} | F1={r['F1']:.4f} | AUC={r['ROC_AUC']:.4f}")
    except Exception as e:
        print(f"ERROR: {e}")
        results.append({"Modelo": name, "Recall": None, "F1": None,
                         "ROC_AUC": None, "Precision": None, "Accuracy": None})

results_df = (pd.DataFrame(results)
              .dropna()
              .sort_values("F1", ascending=False)
              .reset_index(drop=True))

display(results_df.style.hide(axis="index").background_gradient(
    subset=["Recall", "F1", "ROC_AUC", "Precision"], cmap="RdYlGn"
))
print(f"\n★ Mejor modelo por F1: {results_df.iloc[0]['Modelo']}")
print(f"  Recall={results_df.iloc[0]['Recall']:.4f} | F1={results_df.iloc[0]['F1']:.4f} | AUC={results_df.iloc[0]['ROC_AUC']:.4f}")

In [ ]:
# Grafica comparativa de F1
fig = px.bar(
    results_df,
    x="Modelo", y="F1", text="F1",
    title="Comparacion de Modelos -- F1-Score (Validacion Cruzada k=5)",
    template="simple_white",
    color="F1", color_continuous_scale="RdYlGn"
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    xaxis_title="Modelo", yaxis_title="F1-Score",
    yaxis=dict(range=[0, 1]), title_x=0.5, height=500
)
fig.show()

# Grafica multimetrica (prioridad clinica: Recall primero)
metrics_long = results_df.melt(
    id_vars="Modelo",
    value_vars=["Recall", "F1", "ROC_AUC", "Precision"],
    var_name="Metrica", value_name="Valor"
)
fig2 = px.bar(
    metrics_long, x="Modelo", y="Valor", color="Metrica",
    barmode="group",
    title="Comparacion multimetrica por modelo (Recall = prioridad clinica)",
    template="simple_white", height=500
)
fig2.update_layout(title_x=0.5, xaxis_tickangle=-20)
fig2.show()

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 10. Ajuste del Umbral de Decision

El umbral por defecto es **0.5**: si la probabilidad predicha supera 0.5, el modelo clasifica
como *diabetico*. En contexto medico, **bajar el umbral** permite capturar mas diabeticos
a costa de mas falsas alarmas (mayor Recall, menor Precision).

La curva **Precision-Recall** permite elegir el umbral optimo segun la prioridad clinica.
El criterio automatico es maximizar el **F1-Score** sobre todos los umbrales posibles.

</div>

In [ ]:
best_model_name = results_df.iloc[0]["Modelo"]
best_model_pipe = models_cv[best_model_name]
best_model_pipe.fit(X_train, y_train)
y_prob_best   = best_model_pipe.predict_proba(X_test)[:, 1]
y_pred_default = best_model_pipe.predict(X_test)

print(f"Mejor modelo: {best_model_name}")
print(f"\nUmbral 0.50 -- Recall: {recall_score(y_test, y_pred_default):.4f} | "
      f"Precision: {precision_score(y_test, y_pred_default):.4f} | "
      f"F1: {f1_score(y_test, y_pred_default):.4f}")

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_best)
f1_per_thresh = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx   = np.argmax(f1_per_thresh)
best_thresh = thresholds[best_idx]

y_pred_tuned = (y_prob_best >= best_thresh).astype(int)
print(f"Umbral {best_thresh:.2f}  -- Recall: {recall_score(y_test, y_pred_tuned):.4f} | "
      f"Precision: {precision_score(y_test, y_pred_tuned):.4f} | "
      f"F1: {f1_score(y_test, y_pred_tuned):.4f}")

In [ ]:
pr_df = pd.DataFrame({
    "Umbral"   : thresholds,
    "Precision": precisions[:-1],
    "Recall"   : recalls[:-1],
    "F1"       : f1_per_thresh[:-1]
})

fig = go.Figure()
fig.add_trace(go.Scatter(x=pr_df["Umbral"], y=pr_df["Recall"],
                          name="Recall", line=dict(color="#E87D4C", width=2)))
fig.add_trace(go.Scatter(x=pr_df["Umbral"], y=pr_df["F1"],
                          name="F1-Score", line=dict(color="#6CBF6C", width=2)))
fig.add_trace(go.Scatter(x=pr_df["Umbral"], y=pr_df["Precision"],
                          name="Precision", line=dict(color="#4C9BE8", width=2)))
fig.add_vline(x=best_thresh, line_dash="dash", line_color="black",
              annotation_text=f"Umbral optimo F1 = {best_thresh:.2f}")
fig.add_vline(x=0.5, line_dash="dot", line_color="gray",
              annotation_text="Default = 0.50")
fig.update_layout(
    title="Recall, F1 y Precision vs Umbral de Decision",
    xaxis_title="Umbral", yaxis_title="Metrica",
    template="simple_white", title_x=0.5, height=450
)
fig.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, preds, title in zip(
    axes,
    [y_pred_default, y_pred_tuned],
    [f"Umbral default (0.50)", f"Umbral ajustado ({best_thresh:.2f})"]
):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=["Sin diabetes","Diabetes"]
    ).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title, fontweight="bold")
plt.suptitle(f"Impacto del ajuste de umbral -- {best_model_name}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 11. Curva ROC -- Modelo Final

La curva ROC grafica la **Sensibilidad** (Recall) frente a **(1 - Especificidad)** para
todos los umbrales posibles.
- **AUC = 1.0**: discriminacion perfecta
- **AUC = 0.5**: clasificador aleatorio

El **indice de Youden J = Sensibilidad + Especificidad - 1** identifica el umbral que
maximiza la suma de ambas metricas, util cuando se quiere equilibrar deteccion y especificidad.

In [ ]:
fpr, tpr, thresh_roc = roc_curve(y_test, y_prob_best)
roc_auc_val = sklearn_auc(fpr, tpr)

youden_idx       = np.argmax(tpr - fpr)
opt_thresh       = thresh_roc[youden_idx]
opt_fpr, opt_tpr = fpr[youden_idx], tpr[youden_idx]

fig_roc, axes_roc = plt.subplots(1, 2, figsize=(14, 5))

ax = axes_roc[0]
ax.plot(fpr, tpr, color="#4C9BE8", lw=2,
        label="ROC (AUC = {:.4f})".format(roc_auc_val))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Aleatorio (AUC = 0.50)")
ax.scatter([opt_fpr], [opt_tpr], color="red", s=80, zorder=5,
           label="Optimo Youden (umbral={:.2f})".format(opt_thresh))
ax.fill_between(fpr, tpr, alpha=0.08, color="#4C9BE8")
ax.set_xlabel("Tasa de Falsos Positivos (1 - Especificidad)", fontsize=10)
ax.set_ylabel("Sensibilidad (Recall)", fontsize=10)
ax.set_title("Curva ROC -- {}\nAUC = {:.4f}".format(best_model_name, roc_auc_val),
             fontsize=11, fontweight="bold")
ax.legend(fontsize=8, loc="lower right")

ax2 = axes_roc[1]
ax2.axis("off")
lines = [
    "Modelo: " + best_model_name,
    "AUC-ROC: {:.4f}".format(roc_auc_val),
    "",
    "Sensibilidad (umbral Youden): {:.4f}".format(opt_tpr),
    "  De 100 diabeticos detecta ~{:.0f}".format(opt_tpr * 100),
    "",
    "Especificidad (umbral Youden): {:.4f}".format(1 - opt_fpr),
    "  De 100 sanos clasifica bien ~{:.0f}".format((1 - opt_fpr) * 100),
    "",
    "Umbral optimo (Youden J): {:.4f}".format(opt_thresh),
    "",
    "Interpretacion clinica:",
    "  AUC > 0.80 = buena discriminacion.",
    "  Se prioriza Sensibilidad para minimizar",
    "  Falsos Negativos (diabeticos no detectados).",
]
ax2.text(0.05, 0.95, "\n".join(lines), transform=ax2.transAxes,
         verticalalignment="top", fontsize=9,
         bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8),
         fontfamily="monospace")
ax2.set_title("Interpretacion de la Curva ROC", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()
print("AUC-ROC: {:.4f}".format(roc_auc_val))
print("Umbral optimo (Youden): {:.4f}".format(opt_thresh))
print("Sensibilidad en umbral optimo: {:.4f}".format(opt_tpr))
print("Especificidad en umbral optimo: {:.4f}".format(1 - opt_fpr))

---
## 12. Importancia de Variables -- Modelo Final

Los factores de riesgo con mayor peso en la prediccion son clave para guiar
intervenciones clinicas preventivas.

In [ ]:
try:
    importances = best_model_pipe.named_steps["model"].feature_importances_
    imp_df = pd.DataFrame({"Feature": all_features, "Importance": importances})
    imp_df = imp_df.sort_values("Importance", ascending=False).head(21)

    fig = px.bar(
        imp_df.sort_values("Importance"),
        x="Importance", y="Feature",
        orientation="h",
        color="Importance", color_continuous_scale="Reds",
        title=f"Importancia de Variables -- {best_model_name}",
        template="plotly_dark"
    )
    fig.update_layout(height=700, title_x=0.5)
    fig.show()
except AttributeError:
    print(f"El modelo {best_model_name} no expone feature_importances_ directamente.")

---
## 13. Tabla Resumen y Seleccion del Modelo Final

In [ ]:
print("=== Ranking de modelos por F1 (Validacion Cruzada k=5) ===")
display(
    results_df.style
    .hide(axis="index")
    .background_gradient(subset=["Recall","F1","ROC_AUC","Precision"], cmap="RdYlGn")
    .format({"Accuracy":"{:.4f}","Precision":"{:.4f}","Recall":"{:.4f}",
             "F1":"{:.4f}","ROC_AUC":"{:.4f}"})
)

best_row = results_df.iloc[0]
print(f"\nModelo seleccionado: {best_row['Modelo']}")
print(f"  Recall   = {best_row['Recall']:.4f}")
print(f"  F1       = {best_row['F1']:.4f}")
print(f"  ROC_AUC  = {best_row['ROC_AUC']:.4f}")
print(f"  Precision= {best_row['Precision']:.4f}")

### Criterios de seleccion

| Criterio | Consideracion |
|---|---|
| **Recall** | Prioridad clinica maxima: minimizar diabeticos no detectados (falsos negativos) |
| **F1-Score** | Metrica principal de ranking: equilibra Recall y Precision |
| **ROC-AUC** | Capacidad discriminativa global, independiente del umbral |
| **Precision** | Reducir alarmas falsas (no-diabeticos clasificados como diabeticos) |
| **Accuracy** | **No** es criterio de seleccion en datos desbalanceados |

> **Nota clinica:** el ajuste de umbral (Seccion 10) permite mover el balance Recall/Precision
> del modelo ganador segun las necesidades especificas del contexto de screening.

---
## 14. Entrenamiento Final y Serializacion

In [ ]:
final_model = models_cv[best_model_name]
final_model.fit(X_train, y_train)
print(f"Modelo final entrenado: {best_model_name}")
print(f"Datos de entrenamiento: {X_train.shape[0]:,} muestras (70% del dataset)")

In [ ]:
model_path    = MODELS_DIR / "model_classification.joblib"
features_path = MODELS_DIR / "features_classification.joblib"

joblib.dump(final_model,        model_path)
joblib.dump(X.columns.tolist(), features_path)

print(f"Modelo guardado en:    {model_path}")
print(f"Features guardadas en: {features_path}")

# Verificacion de carga
loaded = joblib.load(model_path)
sample_proba = loaded.predict_proba(X_test.iloc[:5])[:, 1]
print(f"\nVerificacion -- probas primeras 5 muestras: {sample_proba.round(4)}")
print("Carga exitosa.")

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## Resumen Ejecutivo

### Estrategias implementadas

| Modelo | Tecnica de balanceo | Mecanismo | Recomendado cuando... |
|---|---|---|---|
| **XGBoost + scale_pos_weight** | Ponderacion interna | Ajusta la funcion de perdida | Los datos son escasos o de alta calidad (sin sinteticos) |
| **SMOTE + XGBoost** | Sobremuestreo sintetico | Genera vecinos interpolados pre-entrenamiento | Se quiere maximizar Recall con dataset balanceado |
| **Balanced Random Forest** | Submuestreo por arbol | Bosque con datos balanceados por arbol | Se necesita interpretabilidad clinica (importancias) |

### Decisiones de diseno

| Decision | Justificacion |
|---|---|
| Sin GridSearch masivo | 253K registros x 5-folds x 20 combinaciones = horas de computo sin ganancia proporcional |
| Hiperparametros directos | Configuraciones probadas en ML medico para ratio ~6:1 |
| Ajuste de umbral | Mayor impacto clinico que tuning fino: mover umbral de 0.5 puede cambiar Recall en 15-20% |
| stratify=y en split | Garantiza misma proporcion de diabeticos en train/test |

**Prioridad de metricas clinicas:** Recall > F1-Score > ROC-AUC > Precision > Accuracy

</div>

---

# Fin del Laboratorio 5

---